# Notebook 03: Develop Judge Prompt

## Learning Goals
- Understand how LLM-as-Judge works
- Learn about few-shot prompting and why it's effective
- Practice selecting good training examples
- Write a clear, effective judge prompt
- Test and iterate on your judge

## What is LLM-as-Judge?

**LLM-as-Judge** means using a Large Language Model (like Claude or GPT-4) to automatically evaluate other AI outputs.

### The Setup:
1. **Input**: A user query + the Recipe Bot's response
2. **Judge's Job**: Determine if the response adheres to dietary restrictions
3. **Output**: PASS or FAIL + reasoning

### Why Use LLMs as Judges?
- **Scales better than humans**: Can evaluate thousands of examples quickly
- **More consistent**: Follows the same criteria every time
- **Cheaper**: Costs less than hiring human evaluators
- **Still needs validation**: That's why we measure TPR/TNR!

## What is Few-Shot Prompting?

**Few-shot prompting** means giving the LLM a few examples of the task before asking it to perform the task.

### Example Structure:
```
You are a judge evaluating dietary adherence.

Example 1:
Query: "vegan pasta recipe"
Response: "Use nutritional yeast instead of cheese"
Label: PASS
Reasoning: No animal products used

Example 2:
Query: "vegan dessert"
Response: "Add honey for sweetness"
Label: FAIL  
Reasoning: Honey is not vegan

Now evaluate this new query...
```

The examples help the LLM understand:
- What level of detail you want in reasoning
- How strict to be about edge cases
- What format to use for output

## Step 1: Load Your Train Set

In [ ]:
import pandas as pd
import json
from pathlib import Path

# Load your train set
train_set = pd.read_csv('../data/train_set.csv')

print(f"Loaded {len(train_set)} training examples")
print(f"\nLabel distribution:")
print(train_set['label'].value_counts())

## Step 2: Understand Dietary Restriction Definitions

Before writing your judge, you need to be clear about what each dietary restriction means.

The README provides definitions - let's load those and make them easily accessible.

In [ ]:
# Dietary restriction definitions from the README
DIETARY_DEFINITIONS = {
    "vegan": "No animal products (meat, dairy, eggs, honey, etc.)",
    "vegetarian": "No meat or fish, but dairy and eggs are allowed",
    "gluten-free": "No wheat, barley, rye, or other gluten-containing grains",
    "dairy-free": "No milk, cheese, butter, yogurt, or other dairy products",
    "keto": "Very low carb (typically <20g net carbs), high fat, moderate protein",
    "paleo": "No grains, legumes, dairy, refined sugar, or processed foods",
    "pescatarian": "No meat except fish and seafood",
    "kosher": "Follows Jewish dietary laws (no pork, shellfish, mixing meat/dairy)",
    "halal": "Follows Islamic dietary laws (no pork, alcohol, proper slaughter)",
    "nut-free": "No tree nuts or peanuts",
    "low-carb": "Significantly reduced carbohydrates (typically <50g per day)",
    "sugar-free": "No added sugars or high-sugar ingredients",
    "raw vegan": "Vegan foods not heated above 118°F (48°C)",
    "whole30": "No grains, dairy, legumes, sugar, alcohol, or processed foods",
    "diabetic-friendly": "Low glycemic index, controlled carbohydrates",
    "low-sodium": "Reduced sodium content for heart health"
}

print("Dietary Restriction Definitions:")
print("=" * 80)
for restriction, definition in DIETARY_DEFINITIONS.items():
    print(f"\n{restriction.upper()}:")
    print(f"  {definition}")

## Step 3: Select Few-Shot Examples from Train Set

Now we need to pick 2-4 examples from the train set to include in our prompt.

### What Makes a Good Few-Shot Example?

1. **Diverse**: Cover different dietary restrictions
2. **Clear**: Obvious PASS or FAIL (not ambiguous)
3. **Representative**: Reflect the kinds of errors the bot makes
4. **Balanced**: Include both PASS and FAIL examples
5. **Instructive**: Show the reasoning style you want

### Selection Strategy:
- At least 1 PASS and 1 FAIL example
- Ideally 2 PASS and 2 FAIL (for 4 total examples)
- Cover different dietary restrictions
- Include at least one "tricky" case if you have one

In [ ]:
# Let's explore what we have available
print("Available examples by label and restriction:\n")

for label in ['PASS', 'FAIL']:
    examples = train_set[train_set['label'] == label]
    print(f"\n{label} Examples ({len(examples)} total):")
    print(examples['dietary_restriction'].value_counts())

In [ ]:
# Browse PASS examples to choose from
print("="*80)
print("BROWSING PASS EXAMPLES")
print("="*80)

pass_examples = train_set[train_set['label'] == 'PASS']

for idx, (i, row) in enumerate(pass_examples.iterrows()):
    print(f"\n--- PASS Example {idx+1} (Index {i}) ---")
    print(f"Restriction: {row['dietary_restriction']}")
    print(f"Query: {row['query'][:100]}...")  
    print(f"Response snippet: {row['response'][:200]}...")
    print(f"Reasoning: {row['reasoning'][:150]}...")
    
    if idx >= 4:  # Show first 5
        print(f"\n... and {len(pass_examples) - 5} more PASS examples")
        break

In [ ]:
# Browse FAIL examples to choose from
print("="*80)
print("BROWSING FAIL EXAMPLES")
print("="*80)

fail_examples = train_set[train_set['label'] == 'FAIL']

for idx, (i, row) in enumerate(fail_examples.iterrows()):
    print(f"\n--- FAIL Example {idx+1} (Index {i}) ---")
    print(f"Restriction: {row['dietary_restriction']}")
    print(f"Query: {row['query'][:100]}...")  
    print(f"Response snippet: {row['response'][:200]}...")
    print(f"Reasoning: {row['reasoning'][:150]}...")
    
    if idx >= 4:  # Show first 5
        print(f"\n... and {len(fail_examples) - 5} more FAIL examples")
        break

### 🎯 Your Task: Select Examples

Based on the examples you browsed above, choose:
- 2 PASS examples
- 2 FAIL examples

Enter their indices in the cell below. Try to pick diverse dietary restrictions!

In [ ]:
# TODO: Replace these with the indices of examples you want to use
# You can find the index numbers in the "Index X" part above

selected_indices = [
    # Add 2 PASS example indices here
    # Example: 0, 5
    
    # Add 2 FAIL example indices here  
    # Example: 2, 8
]

# Load selected examples
selected_examples = train_set.loc[selected_indices]

print(f"Selected {len(selected_examples)} examples:")
print(selected_examples[['dietary_restriction', 'label']].to_string())

## Step 4: Write Your Judge Prompt

Now let's construct your judge prompt. A good judge prompt has these components:

### Components of a Good Judge Prompt:

1. **Role Definition**: Who is the judge?
2. **Task Description**: What are you evaluating?
3. **Evaluation Criteria**: What makes something PASS vs FAIL?
4. **Dietary Definitions**: Clear definitions of each restriction
5. **Few-Shot Examples**: Your selected examples
6. **Output Format**: What format should the response be in? (JSON is recommended)
7. **Edge Case Handling**: How to handle ambiguity

Let's build it step by step!

In [ ]:
# Helper function to create few-shot example text
def format_few_shot_example(row, example_num):
    """Format a training example for inclusion in the prompt."""
    return f"""### Example {example_num}:
**User Query:** {row['query']}
**Dietary Restriction:** {row['dietary_restriction']}
**Recipe Bot Response:**
{row['response']}

**Evaluation:**
{{
  "label": "{row['label']}",
  "reasoning": "{row['reasoning']}"
}}
"""

# Generate few-shot examples text
few_shot_text = ""
for i, (idx, row) in enumerate(selected_examples.iterrows(), 1):
    few_shot_text += format_few_shot_example(row, i)
    few_shot_text += "\n" + "-"*80 + "\n\n"

print("Few-shot examples formatted!")
print(f"Generated {len(selected_examples)} examples")

In [ ]:
# Build the complete judge prompt
judge_prompt = f"""# Recipe Bot Dietary Adherence Judge

## Your Role
You are an expert nutritionist and dietary restriction specialist. Your job is to evaluate whether a Recipe Bot's response correctly adheres to a user's specified dietary restrictions.

## Task
Given:
1. A user query requesting a recipe with specific dietary restrictions
2. The Recipe Bot's response
3. The dietary restriction to evaluate

You must determine:
- **PASS**: The recipe fully adheres to the dietary restriction
- **FAIL**: The recipe violates the dietary restriction in any way

## Evaluation Criteria

A response should be labeled **PASS** if:
- All ingredients comply with the dietary restriction
- All preparation methods comply with the restriction (e.g., raw vegan must not cook above 118°F)
- Any substitutions suggested are appropriate for the restriction
- The recipe doesn't assume ingredients that would violate the restriction

A response should be labeled **FAIL** if:
- ANY ingredient violates the restriction (even as an optional ingredient)
- Preparation methods violate the restriction
- The recipe suggests non-compliant substitutions
- The recipe is ambiguous about compliance (err on the side of FAIL for ambiguity)

## Dietary Restriction Definitions

{''.join([f'- **{k}**: {v}\n' for k, v in DIETARY_DEFINITIONS.items()])}

## Important Notes

- Be **strict** in your evaluation - even small violations are failures
- Check **all ingredients**, including garnishes, seasonings, and optional toppings
- Consider **hidden ingredients** (e.g., soy sauce contains wheat, many broths contain dairy)
- If a recipe is **ambiguous** or unclear, label as FAIL and explain the ambiguity
- Provide **specific reasoning** citing which ingredients or methods caused the pass/fail

## Few-Shot Examples

{few_shot_text}

## Output Format

You MUST respond with valid JSON in exactly this format:

{{
  "label": "PASS" or "FAIL",
  "reasoning": "Detailed explanation citing specific ingredients or methods"
}}

## Now Evaluate This Query

**User Query:** {{query}}
**Dietary Restriction:** {{dietary_restriction}}
**Recipe Bot Response:**
{{response}}

**Your Evaluation:**
"""

print("✅ Judge prompt created!")
print(f"\nPrompt length: {len(judge_prompt)} characters")
print(f"\nPreview (first 500 characters):")
print(judge_prompt[:500])

## Step 5: Save Your Prompt

Let's save this prompt so you can use it in later notebooks and keep track of versions.

In [ ]:
# Save the prompt
prompt_dir = Path('../prompts')
prompt_dir.mkdir(exist_ok=True)

prompt_path = prompt_dir / 'judge_prompt_v1.txt'

with open(prompt_path, 'w') as f:
    f.write(judge_prompt)

print(f"✅ Prompt saved to: {prompt_path}")
print("\nYou can create new versions (v2, v3) as you iterate!")

## Step 6: Test Your Judge Manually

Before we run the judge on the full dev set, let's test it manually on a few examples.

### How to Test:
1. Load a few examples from your **dev set** (NOT train!)
2. Fill in the prompt template with real data
3. Copy the filled prompt
4. Paste into Claude/ChatGPT interface
5. See if the judge's answer matches the ground truth
6. Iterate on your prompt if needed

**Important**: This is a manual step! You'll copy-paste into Claude or ChatGPT yourself.

In [ ]:
# Load dev set for testing
dev_set = pd.read_csv('../data/dev_set.csv')
print(f"Loaded {len(dev_set)} dev examples for testing")

In [ ]:
# Select a test example from dev set
test_example = dev_set.iloc[0]  # Change index to test different examples

print("=" * 80)
print("TEST EXAMPLE")
print("=" * 80)
print(f"\nDietary Restriction: {test_example['dietary_restriction']}")
print(f"Query: {test_example['query']}")
print(f"\nResponse: {test_example['response'][:300]}...")
print(f"\nGround Truth Label: {test_example['label']}")
print(f"Ground Truth Reasoning: {test_example['reasoning']}")

In [ ]:
# Fill in the prompt template with this example
# Using .replace() instead of .format() to avoid conflicts with JSON braces
filled_prompt = judge_prompt.replace("{query}", test_example['query'])
filled_prompt = filled_prompt.replace("{dietary_restriction}", test_example['dietary_restriction'])
filled_prompt = filled_prompt.replace("{response}", test_example['response'])

print("=" * 80)
print("FILLED PROMPT - COPY THIS INTO CLAUDE/CHATGPT")
print("=" * 80)
print(filled_prompt)
print("=" * 80)

### 🧪 Testing Instructions:

1. **Copy** the filled prompt above
2. **Open** Claude.ai or ChatGPT in a new browser tab
3. **Paste** the prompt and submit
4. **Compare** the judge's output with the ground truth label above
5. **Evaluate**:
   - Did the judge get the label right (PASS vs FAIL)?
   - Is the reasoning clear and correct?
   - Is the output in valid JSON format?

### Test Multiple Examples:
- Change `test_example = dev_set.iloc[0]` to `iloc[1]`, `iloc[2]`, etc.
- Try at least 3-5 examples
- Test both PASS and FAIL cases
- Test different dietary restrictions

### Record Your Results:
In the cell below, document how your judge performed.

### Your Manual Testing Results:

**Test 1 (Example index X):**
- Ground truth: [PASS/FAIL]
- Judge said: [PASS/FAIL]
- Correct? [Yes/No]
- Notes: [Your observations]

**Test 2 (Example index Y):**
- Ground truth: [PASS/FAIL]
- Judge said: [PASS/FAIL]
- Correct? [Yes/No]
- Notes: [Your observations]

[Add more tests as needed]

**Overall Observations:**
- What patterns did you notice?
- Did the judge make any consistent mistakes?
- Does the prompt need improvement?
- Are the few-shot examples helping?

## Step 7: Iterate on Your Prompt (Optional)

Based on your manual testing, you might want to improve your prompt.

### Common Issues and Fixes:

1. **Judge is too lenient (false positives)**:
   - Add emphasis on being strict
   - Add FAIL examples showing subtle violations
   - Clarify specific ingredients to watch for

2. **Judge is too strict (false negatives)**:
   - Clarify when substitutions are acceptable
   - Add PASS examples with good substitutions
   - Explain the difference between strict and reasonable interpretation

3. **Reasoning is unclear**:
   - Update few-shot examples to show better reasoning
   - Add explicit instruction for detailed reasoning

4. **Output format issues**:
   - Emphasize JSON format requirement
   - Show exact JSON structure in examples

### If you want to iterate:
1. Modify the prompt in Step 4 above
2. Save as a new version (v2, v3, etc.)
3. Test again with the same dev examples
4. Compare performance across versions

## Summary: What You've Learned

### Key Concepts:
1. **LLM-as-Judge**: Using LLMs to automatically evaluate AI outputs
2. **Few-Shot Prompting**: Providing examples to guide the LLM's behavior
3. **Prompt Engineering**: Crafting clear, effective instructions
4. **Iterative Development**: Test, observe, refine, repeat

### What You Created:
- ✅ A complete judge prompt with:
  - Clear role definition
  - Evaluation criteria
  - Dietary restriction definitions
  - 4 few-shot examples from your train set
  - Structured JSON output format

### Best Practices:
1. **Use train set for examples** - Never use dev/test examples in your prompt
2. **Test manually first** - Understand your judge before automating
3. **Be specific** - Clear criteria prevent ambiguity
4. **Show, don't just tell** - Few-shot examples are more powerful than instructions alone
5. **Version your prompts** - Track changes as you iterate

### Next Steps:
Now that you have a working judge prompt, you're ready for **Notebook 04: Evaluate Judge Performance**!

In the next notebook, you'll:
- Run your judge on the entire dev set programmatically
- Calculate confusion matrices
- Measure TPR and TNR
- Analyze where your judge succeeds and fails
- Run final evaluation on test set